<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>实验：数据增强</h1> 


<h2>目录</h2>

在本实验中，我们将使用常规数据和增强数据训练卷积神经网络。本实验的目的是展示增强数据可以提高泛化性能。

设想一个场景：无人机必须拍摄某个物体的照片。无人机在移动，物体也可能在移动。拍摄图像时，我们并不总是能得到完美的图像。被摄物体可能不在图像正中央，或者在图像中发生了旋转。在这种情况下，一个在完美居中或未经旋转的图像上训练的模型表现不会很好。因此，我们在旋转数据上训练模型，使其在不完美的图像上也能表现良好。

在本作业中，我们将使用一个数字图像数据集。我们将训练两个模型：一个在非旋转数字上训练，另一个在旋转图像上训练；然后在旋转测试数据集上测试这两个模型，这相对于我们上面的场景来说更加真实和稳健。

- [加载数据](#Load-Data)
- [构建卷积神经网络类](#Build-a-Convolutional-Neural-Network-Class)
- [常规数据](#Regular-Data)
- [旋转训练数据](#Rotated-Training-Data)

<p>预计所需时间：<strong>25 分钟</strong> </p>

<hr>


<h2>准备工作</h2>


In [ ]:
%%time
%pip install numpy matplotlib Pillow==6.2.2
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
--index-url https://download.pytorch.org/whl/cpu


In [ ]:
# 下载预训练模型，因为训练需要很长时间
!wget https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/meet_up/12.02.2020/normal.pt


In [ ]:
# 下载在增强数据上训练的预训练模型，因为训练需要很长时间
!wget https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/meet_up/12.02.2020/rotated_data.pt


***重启内核***


In [ ]:
# 导入本实验需要使用的库
# 使用以下代码安装 torchvision 库
# !conda install -y torchvision

# PyTorch 库
import torch 
# PyTorch 神经网络库
import torch.nn as nn
# 允许我们转换数据
import torchvision.transforms as transforms
# 用于绘制数据和损失曲线
import matplotlib.pylab as plt
# 允许我们使用数组来操作和存储数据
import numpy as np
# 允许我们下载数据集
import torchvision.datasets as dsets
# 允许我们访问文件系统
import os  


一些用于绘图的实用函数


绘制代价和准确率随轮次变化的图


In [ ]:
def plot_cost_accuracy(checkpoint):

# 绘制代价和准确率

    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(checkpoint['cost'], color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('Cost', color=color)
    ax1.tick_params(axis='y', color=color)
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color) 
    ax2.set_xlabel('epoch', color=color)
    ax2.plot( checkpoint['accuracy'], color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()


定义 <code>show_data</code> 函数将数据样本绘制为图像。


In [ ]:
def show_data(data_sample):
    plt.imshow(data_sample[0].numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    plt.title('y = '+ str(data_sample[1]))


绘制前 5 个分类错误的样本


In [ ]:
def plot_mis_classified(model, dataset):
    count=0
    for x, y in torch.utils.data.DataLoader(dataset=dataset, batch_size=1):
        z = model(x)
        _, yhat = torch.max(z, 1)
        if yhat != y:
            show_data((x, y))
            plt.show()
            count += 1
        if count >= 5:
            break 


<!--Empty Space for separating topics-->


<h2 id="Makeup_Data">加载数据</h2> 


我们创建一个变换对象 <code>compose</code>，其中一个用于调整图像大小并将其转换为张量，另一个还会随机旋转图像。


In [ ]:
# 图像大小为 16×16
IMAGE_SIZE = 16

# 创建一组变换以生成旋转数据集
# 调整图像大小，随机旋转，然后转换为张量
compose_rotate = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),transforms.RandomAffine(45), transforms.ToTensor()])

# 创建一组变换以生成非旋转数据集
# 调整图像大小，然后转换为张量
compose = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])


通过将参数 <code>train</code> 设置为 <code>True</code> 来加载训练数据集。我们使用上面定义的变换：一个用于旋转数据，一个用于非旋转数据。


In [ ]:
# 变换参数设置为对应的组合
train_dataset_rotate = dsets.MNIST(root='./data', train=True, download=True, transform=compose_rotate)
train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=compose)


通过将参数 train 设置为 <code>False</code> 来加载测试数据集，其中数据<b>全部</b>经过旋转。


In [ ]:
# 加载测试数据集
validation_dataset = dsets.MNIST(root='./data', train=False, download=True, transform=compose_rotate)


矩形张量中的每个元素对应一个表示像素强度的数字，如下图所示。


<img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0110EN/notebook_images%20/chapter%206/6.2.1imagenet.png" width="550" alt="MNIST data image">


绘制第一个样本


In [ ]:
# 第一个数据样本的图像
show_data(train_dataset[0])


输出第一个标签，我们可以看到这个数字是 5


In [ ]:
# 第一个数据元素的标签
train_dataset[0][1]


现在让我们绘制旋转训练数据集的第一个样本


In [ ]:
show_data(train_dataset_rotate[0])


我们可以看到这是同一个样本，但它按我们期望的方式进行了旋转


<!--Empty Space for separating topics-->


<h2 id="CNN">构建卷积神经网络类</h2>


构建一个具有两个卷积层和一个全连接层的卷积网络类。预先确定最终输出矩阵的大小。构造函数中的参数是第一层和第二层的输出通道数。


In [ ]:
class CNN(nn.Module):
    
    # 构造函数
    def __init__(self, out_1=16, out_2=32):
        super(CNN, self).__init__()
        
        # 我们从 1 个通道开始，因为图像是单通道的黑白图像
        # 该层后的通道宽度为 16
        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=out_1, kernel_size=5, padding=2)
        # 该层后的通道宽度为 8
        self.maxpool1=nn.MaxPool2d(kernel_size=2)
        
        # 该层后的通道宽度为 8
        self.cnn2 = nn.Conv2d(in_channels=out_1, out_channels=out_2, kernel_size=5, stride=1, padding=2)
        # 该层后的通道宽度为 4
        self.maxpool2=nn.MaxPool2d(kernel_size=2)
        # 根据上面的宽度计算，我们总共有 out_2 (32) 个通道，每个通道大小为 4×4。通道是正方形的。
        # 输出是每个类别的值
        self.fc1 = nn.Linear(out_2 * 4 * 4, 10)
    
    # 预测
    def forward(self, x):
        # 将 X 值通过每个 CNN、ReLU 和池化层，然后将其展平以输入全连接层
        x = self.cnn1(x)
        x = torch.relu(x)
        x = self.maxpool1(x)
        x = self.cnn2(x)
        x = torch.relu(x)
        x = self.maxpool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

    # 输出 CNN、ReLU 和池化层每个阶段的结果
    def activations(self, x):
        # 输出激活图，这不是必需的
        z1 = self.cnn1(x)
        a1 = torch.relu(z1)
        out = self.maxpool1(a1)
        
        z2 = self.cnn2(out)
        a2 = torch.relu(z2)
        out1 = self.maxpool2(a2)
        out = out.view(out.size(0),-1)
        return z1, a1, z2, a2, out1,out


<h2 id="regular_data">常规数据</h2> 


定义卷积神经网络分类器、损失函数、优化器并训练模型


In [ ]:
# 使用 CNN 类创建将在常规数据上训练的模型对象
model = CNN(out_1=16, out_2=32)


定义损失函数、优化器和数据加载器


In [ ]:
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
learning_rate = 0.1
# 创建一个使用学习率和梯度更新模型参数的优化器
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)
# 为训练数据创建一个批量大小为 100 的数据加载器 
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=100)
# 为旋转验证数据创建一个批量大小为 5000 的数据加载器 
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)


该单元格将训练模型，由于运行时间较长，我们将其注释掉。你可以将块类型从 Raw 改为 Code 并运行它，也可以在下一个单元格中加载训练好的模型。请注意，我们在这里不仅训练和保存模型，还会跟踪训练过程中的重要数据，例如代价和准确率。


In [ ]:
# 训练模型
import os  

# 保存数据的位置
file_normal = os.path.join(os.getcwd(), 'normal.pt')

# 我们保存的所有数据
checkpoint={
            # 保存模型训练的轮次数
            'epoch': None,
            # 保存模型参数，以便我们可以重新创建训练好的模型
            'model_state_dict': None,
            # 保存优化器参数
            'optimizer_state_dict': None,
            # 保存训练数据集上最后一轮最后一个批次的损失
            'loss': None,
            # 保存训练数据集上每个轮次的代价
            'cost': [],
            # 保存测试数据集上每个轮次的准确率
            'accuracy': []}
            
# 训练模型的轮次数
n_epochs = 5

# 测试数据集的大小
N_test = len(validation_dataset)

# 按我们想要的轮次数进行训练
for epoch in range(n_epochs):
    # 用于跟踪每个轮次代价的变量
    cost = 0
    # 遍历训练数据集中的每个批次
    for x, y in train_loader:
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置，梯度会累积
        optimizer.zero_grad()
        # 对图像进行预测
        z = model(x)
        # 计算预测值与实际类别之间的损失
        loss = criterion(z, y)
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
      
        # 保存我们训练的轮次数  
        checkpoint['epochs'] = n_epochs
        # 保存模型参数
        checkpoint['model_state_dict'] = model.state_dict()
        # 保存优化器参数
        checkpoint['optimizer_state_dict'] = optimizer.state_dict()
        # 保存最后一个批次的损失，因此最终将是最后一轮最后一个批次的损失
        checkpoint['loss'] = loss
        # 累积损失
        cost += loss.item()
        
     
    # 正确预测数量的计数器        
    correct = 0
        
    # 遍历验证数据集中的每个批次
    for x_test, y_test in validation_loader:
        # 进行预测
        z = model(x_test)
        # 获取具有最大值的类别
        _, yhat = torch.max(z.data, 1)
        # 统计正确预测的数量
        correct += (yhat == y_test).sum().item()
 
    accuracy = correct / N_test
    print(accuracy)
    # 将该轮次的代价追加到列表中
    checkpoint['cost'].append(cost) 
    # 将该轮次的准确率追加到列表中
    checkpoint['accuracy'].append(accuracy)
    # 将检查点中的数据保存到文件位置
    torch.save(checkpoint, file_normal) 


<h3 id="Result">分析结果</h3> 


加载保存在 `normal.pt` 中的数据


In [ ]:
checkpoint_normal = torch.load(os.path.join(os.getcwd(),'normal.pt'))


绘制准确率和代价随轮次变化的图


In [ ]:
# 使用顶部定义的辅助函数以及我们保存的代价和准确率列表
plot_cost_accuracy(checkpoint_normal)


五个分类错误的样本


In [ ]:
# 使用我们保存的模型参数加载到模型中，以重新创建训练好的模型
model.load_state_dict(checkpoint_normal['model_state_dict'])
# 将模型设置为评估模式
model.eval()
# 使用辅助函数绘制前五个分类错误的样本
plot_mis_classified(model,validation_dataset)


<!--Empty Space for separating topics-->


<h2 id="R_training_data">旋转训练数据</h2> 


绘制验证数据上的损失和准确率：


In [ ]:
# 使用 CNN 类创建模型对象
model_r = CNN(out_1=16, out_2=32)
# 创建一个用于测量损失的准则
criterion = nn.CrossEntropyLoss()
learning_rate = 0.1
# 创建一个使用学习率和梯度更新模型参数的优化器
optimizer = torch.optim.SGD(model_r.parameters(), lr = learning_rate)
# 为旋转训练数据创建一个批量大小为 100 的数据加载器 
train_loader = torch.utils.data.DataLoader(dataset=train_dataset_rotate, batch_size=100)
# 为旋转验证数据创建一个批量大小为 5000 的数据加载器 
validation_loader = torch.utils.data.DataLoader(dataset=validation_dataset, batch_size=5000)


该单元格将训练模型，由于运行时间较长，我们将其注释掉。你可以将块类型从 Raw 改为 Code 并运行它，也可以在下一个单元格中加载训练好的模型。


In [ ]:
# 保存数据的位置
file_rotated = os.path.join(os.getcwd(), 'rotated_data.pt')

# 我们保存的所有数据
checkpoint={
            # 保存模型训练的轮次数
            'epoch': None,
            # 保存模型参数，以便我们可以重新创建训练好的模型
            'model_state_dict': None,
            # 保存优化器参数
            'optimizer_state_dict': None,
            # 保存训练数据集上最后一轮最后一个批次的损失
            'loss': None,
            # 保存训练数据集上每个轮次的代价
            'cost': [],
            # 保存测试数据集上每个轮次的准确率
            'accuracy': []}
            
# 训练模型的轮次数            
n_epochs = 5

# 测试数据集的大小
N_test = len(validation_dataset)

# 按我们想要的轮次数进行训练
for epoch in range(n_epochs):
    # 用于跟踪每个轮次代价的变量
    cost = 0
    # 遍历训练数据集中的每个批次
    for x, y in train_loader:
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置，梯度会累积
        optimizer.zero_grad()
        # 对图像进行预测
        z = model_r(x)
        # 计算预测值与实际类别之间的损失
        loss = criterion(z, y)
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
      
        # 保存我们训练的轮次数  
        checkpoint['epochs'] = n_epochs
        # 保存模型参数
        checkpoint['model_state_dict'] = model.state_dict()
        # 保存优化器参数
        checkpoint['optimizer_state_dict'] = optimizer.state_dict()
        # 保存最后一个批次的损失，因此最终将是最后一轮最后一个批次的损失
        checkpoint['loss'] = loss
        # 累积损失
        cost+=loss.item()
        
     
    # 正确预测数量的计数器        
    correct = 0
        
    # 遍历验证数据集中的每个批次
    for x_test, y_test in validation_loader:
        # 进行预测
        z = model_r(x_test)
        # 获取具有最大值的类别
        _, yhat = torch.max(z.data, 1)
        # 统计正确预测的数量
        correct += (yhat == y_test).sum().item()
 
    accuracy = correct / N_test
    print(accuracy)
    # 将该轮次的代价追加到列表中
    checkpoint['cost'].append(cost) 
    # 将该轮次的准确率追加到列表中
    checkpoint['accuracy'].append(accuracy)
    # 将检查点中的数据保存到文件位置
    torch.save(checkpoint, file_rotated) 


<h3 id="Result">分析结果</h3> 


加载保存在 `rotated_data.pt` 中的数据


In [ ]:
checkpoint_rotated= torch.load(os.path.join(os.getcwd(),'rotated_data.pt'))


绘制准确率和代价随轮次变化的图


In [ ]:
# 使用顶部定义的辅助函数以及我们保存的代价和准确率列表
plot_cost_accuracy(checkpoint_rotated)


五个分类错误的样本


In [ ]:
# 使用我们保存的模型参数加载到模型中，以重新创建训练好的模型
model_r.load_state_dict(checkpoint_rotated['model_state_dict'])
# 将模型设置为评估模式
model.eval()
# 使用辅助函数绘制前五个分类错误的样本
plot_mis_classified(model_r,validation_dataset)


## 总结


正如你在两个代价和准确率随轮次变化的图中所见，在旋转数据上训练的模型在旋转验证数据上表现得更好，准确率 79% 对比 95%+。通过在旋转数据上创建和训练，模型能够从独特的数据中学习，并更多地接触现实生活中的情况，因为数据并不总是完美或正确旋转的。


<!--Empty Space for separating topics-->


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，他的研究重点是利用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自获得博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>


感谢 Magnus <a href="http://www.hvass-labs.org/">Erik Hvass Pedersen</a>，他的教程帮助我理解卷积神经网络


<!--## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-23  | 2.0  | Sathya  |  Converted lab to Jupyterlab Current|-->


## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>
